<a href="https://colab.research.google.com/github/dongYoun2/DL-HW/blob/main/hw4/hw4_impl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Vision Transformer (ViT)

In this assignment we're going to work with Vision Transformer. We will start to build our own vit model and train it on an image classification task.
The purpose of this homework is for you to get familar with ViT and get prepared for the final project.

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# VIT Implementation

The vision transformer can be seperated into three parts, we will implement each part and combine them in the end.

For the implementation, feel free to experiment different kinds of setup, as long as you use attention as the main computation unit and the ViT can be train to perform the image classification task present later.
You can read about the ViT implement from other libary: https://github.com/huggingface/pytorch-image-models/blob/main/timm/models/vision_transformer.py and https://github.com/pytorch/vision/blob/main/torchvision/models/vision_transformer.py

## PatchEmbedding
PatchEmbedding is responsible for dividing the input image into non-overlapping patches and projecting them into a specified embedding dimension. It uses a 2D convolution layer with a kernel size and stride equal to the patch size. The output is a sequence of linear embeddings for each patch.

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, image_size, patch_size, in_channels, embed_dim):
        # TODO
        super().__init__()
        assert image_size % patch_size == 0, "image_size must be divisible by patch_size"
        self.image_size = image_size
        self.patch_size = patch_size
        self.in_channels = in_channels
        self.embed_dim = embed_dim

        # Compute number of patches
        self.num_patches = (image_size // patch_size) ** 2

        # Each patch (patch_size × patch_size) → embedding of dim embed_dim
        self.projection = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        # TODO
        """
        Args:
            x: Tensor of shape [B, C, H, W]
        Returns:
            Tensor of shape [B, N, E] where
                N = num_patches, E = embed_dim
        """
        x = self.projection(x)          # [B, E, H/ps, W/ps]
        x = x.flatten(2).transpose(1, 2)  # [B, N, E]
        return x

## MultiHeadSelfAttention

This class implements the multi-head self-attention mechanism, which is a key component of the transformer architecture. It consists of multiple attention heads that independently compute scaled dot-product attention on the input embeddings. This allows the model to capture different aspects of the input at different positions. The attention outputs are concatenated and linearly transformed back to the original embedding size.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        # TODO
        super(MultiHeadSelfAttention, self).__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # Per-head projections (kept as separate layers to preserve the original API/attributes)
        self.query_projection = nn.Linear(embed_dim, embed_dim)
        self.key_projection   = nn.Linear(embed_dim, embed_dim)
        self.value_projection = nn.Linear(embed_dim, embed_dim)

        # Output projection after concatenating heads
        self.final_projection = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        # TODO
        """
        Args:
            x: Tensor [B, N, E] where
                B = batch size, N = sequence length (num_patches), E = embed_dim
        Returns:
            Tensor [B, N, E]
        """
        B, N, _ = x.size()

        # 1) Linear projections → [B, N, E]
        q = self.query_projection(x)
        k = self.key_projection(x)
        v = self.value_projection(x)

        # 2) Split heads → [B, H, N, D]
        def split_heads(t):
            return t.reshape(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        q, k, v = map(split_heads, (q, k, v))

        # 3) Scaled dot-product attention (uses efficient kernel under the hood)
        #    Output: [B, H, N, D]
        attn = F.scaled_dot_product_attention(q, k, v)

        # 4) Merge heads → [B, N, E]
        attn = attn.transpose(1, 2).contiguous().reshape(B, N, self.embed_dim)

        # 5) Final projection
        return self.final_projection(attn)

## TransformerBlock
This class represents a single transformer layer. It includes a multi-head self-attention sublayer followed by a position-wise feed-forward network (MLP). Each sublayer is surrounded by residual connections.
You may also want to use layer normalization or other type of normalization.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout):
        # TODO
        super(TransformerBlock, self).__init__()
        self.attention = MultiHeadSelfAttention(embed_dim, num_heads)

        # Post-attention: dropout + norm (keep original post-norm ordering)
        self.attn_dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(embed_dim)

        # Feed-forward network (MLP)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout),
        )

        # Post-MLP norm (keep original)
        self.norm2 = nn.LayerNorm(embed_dim)


    def forward(self, x):
        # TODO
        """
        Args:
            x: Tensor [B, N, E] — batch, sequence length, embedding dim
        Returns:
            Tensor [B, N, E]
        """
        # 1) Multi-head self-attention + residual + norm
        attn_out = self.attention(x)                # [B, N, E]
        x = x + self.attn_dropout(attn_out)         # residual
        x = self.norm1(x)                           # post-norm

        # 2) Position-wise MLP + residual + norm
        mlp_out = self.mlp(x)                       # [B, N, E]
        x = x + mlp_out                             # residual
        x = self.norm2(x)                           # post-norm
        return x

## VisionTransformer:
This is the main class that assembles the entire Vision Transformer architecture. It starts with the PatchEmbedding layer to create patch embeddings from the input image. A special class token is added to the sequence, and positional embeddings are added to both the patch and class tokens. The sequence of patch embeddings is then passed through multiple TransformerBlock layers. The final output is the logits for all classes

In [ ]:
from torch.nn.init import trunc_normal_


class VisionTransformer(nn.Module):
    def __init__(self, image_size, patch_size, in_channels, embed_dim, num_heads, mlp_dim, num_layers, num_classes, dropout=0.1):
        # TODO
        super().__init__()
        assert image_size % patch_size == 0, "image_size must be divisible by patch_size"

        # Patch/token embeddings
        self.patch_embedding = PatchEmbedding(image_size, patch_size, in_channels, embed_dim)

        # [CLS] token and positional embeddings (learnable)
        self.class_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        num_patches = (image_size // patch_size) ** 2
        self.positional_embedding = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))

        # Transformer encoder blocks
        self.transformer_blocks = nn.ModuleList(
            [TransformerBlock(embed_dim, num_heads, mlp_dim, dropout) for _ in range(num_layers)]
        )

        # Optional token/dropout before entering encoder (keeps API; improves regularization)
        self.dropout = nn.Dropout(dropout)

        # Head
        self.classification_head = nn.Linear(embed_dim, num_classes)

        # Init
        self._reset_parameters()

    def _reset_parameters(self):
      # ViT-style initialization
      trunc_normal_(self.class_token, std=0.02)
      trunc_normal_(self.positional_embedding, std=0.02)
      trunc_normal_(self.classification_head.weight, std=0.02)
      if self.classification_head.bias is not None:
          nn.init.zeros_(self.classification_head.bias)

    def forward(self, x):
        # TODO
        """
        Args:
            x: Tensor [B, C, H, W]
        Returns:
            logits: Tensor [B, num_classes]
        """
        B = x.size(0)

        # 1) Patch embedding -> [B, N, E]
        x = self.patch_embedding(x)

        # 2) Prepend [CLS] token
        cls = self.class_token.expand(B, -1, -1)      # [B, 1, E]
        x = torch.cat((cls, x), dim=1)                # [B, N+1, E]

        # 3) Add positional embeddings (safe slice) + dropout
        x = x + self.positional_embedding[:, : x.size(1), :]
        x = self.dropout(x)

        # 4) Transformer encoder
        for blk in self.transformer_blocks:
            x = blk(x)

        # 5) Classification on [CLS] token
        cls_token_out = x[:, 0]                       # [B, E]
        logits = self.classification_head(cls_token_out)
        return logits

## Let's train the ViT!

We will train the vit to do the image classification with cifar100. Free free to change the optimizer and or add other tricks to improve the training

In [ ]:
# Example usage:
# TODO
image_size = 64
patch_size = 8
in_channels = 3
embed_dim = 256
num_heads = 4
mlp_dim = 512
num_layers = 8
num_classes = 100
dropout = 0.1
batch_size = 128

max_grad_norm = 1
base_lr = 5e-4
weight_decay = 0.05
num_epochs = 1  # TODO

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

# Load the CIFAR-100 dataset
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.Resize(image_size),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2), ratio=(0.3, 3.3)),  # cheap regularizer
])

transform_test = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

trainset = datasets.CIFAR100(root='./data', train=True, download=True, transform=transform_train)
testset = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_test)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
def save_checkpoint(path, model, optimizer, scheduler, epoch: int, best_val_acc: float):
    state = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "epoch": epoch,
        "best_val_acc": best_val_acc,
    }
    torch.save(state, path)


def load_checkpoint(path, model, optimizer, scheduler):
    ckpt = torch.load(path, map_location=device, weights_only=False)

    model.load_state_dict(ckpt["model"], strict=True)
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])

    start_epoch: int = ckpt["epoch"]
    best_val_acc: float = ckpt["best_val_acc"]


    return model, optimizer, scheduler, start_epoch, best_val_acc

In [ ]:
from pathlib import Path

# Train the model
best_ckpt_path = "best_ckpt.pt"
last_ckpt_path = "last_ckpt.pt"

model = VisionTransformer(image_size, patch_size, in_channels, embed_dim, num_heads, mlp_dim, num_layers, num_classes, dropout).to(device)
input_tensor = torch.randn(1, in_channels, image_size, image_size).to(device)
output = model(input_tensor)
print("Checking output shape: ", output.shape)


# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-4) # TODO
optimizer = torch.optim.AdamW(model.parameters(), lr=base_lr, weight_decay=weight_decay) # TODO

num_steps_per_epoch = len(trainloader)
total_steps = num_epochs * num_steps_per_epoch
warmup_epochs = 5
warmup_steps = warmup_epochs * num_steps_per_epoch

# Warm-up to base_lr, then cosine down to ~0
warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1e-3, end_factor=1.0, total_iters=warmup_steps)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=(total_steps - warmup_steps), eta_min=1e-6)

scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_steps])


if Path(last_ckpt_path).exists():
    model, optimizer, scheduler, last_epoch, best_val_acc = load_checkpoint(last_ckpt_path, model, optimizer, scheduler)
    start_epoch = last_epoch + 1
    print(f"Loaded checkpoint from: {last_ckpt_path} | resume after epoch {last_epoch + 1}, best_val_acc={best_val_acc:.2f}%")

else:
    start_epoch = 0
    best_val_acc = 0

best_val_acc = 0
for epoch in range(start_epoch, start_epoch + num_epochs):
    model.train()
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()

        # Add gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)

        optimizer.step()
        scheduler.step()

        # TODO Feel free to modify the training loop youself.

    # Validate the model
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in testloader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    val_acc = 100 * correct / total
    print(f"Epoch: {epoch + 1}, Validation Accuracy: {val_acc:.2f}%")


    # Save the best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint(best_ckpt_path, model, optimizer, scheduler, epoch, best_val_acc)

    # save the last model
    save_checkpoint(last_ckpt_path, model, optimizer, scheduler, epoch, best_val_acc)

Please submit your best_model.pth with this notebook. And report the best test results you get.

In [ ]:
model.load_state_dict(torch.load(best_ckpt_path, map_location=device)["model"])
model.to(device)

# Validation loop
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
val_acc = 100 * correct / total

print(f"Test Accuracy: {val_acc:.2f}%")

added:
1. additional data augmentation: random erasing
2. gradient clipping
3. apply scheduler: warmup, then cosine (per-iteration)